# Data Splitting

## Objective

In this notebook, we will practice:

- Separating Features and Target
- Train/Test Split
- Validation Set
- Stratified Sampling
- Cross Validation
- Stratified K-Fold
- Data Leakage Prevention

> A randomly generated `practice_target` is used only to learn the Data Splitting workflow.
> It does not represent a real medical outcome.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold
)

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [2]:
df = pd.read_csv(
    "../heart_failure_clinical_records_dataset-selected-columns.csv"
)

df.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex
0,75.0,0,582,0,20,1,265000.00,1.9,130,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0


In [3]:
np.random.seed(42)

df_practice = df.copy()

df_practice["practice_target"] = np.random.randint(
    0,
    2,
    size=len(df_practice)
)

df_practice["practice_target"].value_counts()

practice_target
0    151
1    148
Name: count, dtype: int64

# Features and Target

Before splitting the dataset:

- `X` contains input features.
- `y` contains the target variable.

In [4]:
X = df_practice.drop(
    columns=["practice_target"]
)

y = df_practice["practice_target"]

print("X Shape:", X.shape)
print("y Shape:", y.shape)

X Shape: (299, 10)
y Shape: (299,)


# Train/Test Split

The dataset will be divided into:

- 80% Training Data
- 20% Testing Data

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (239, 10)
X_test: (60, 10)
y_train: (239,)
y_test: (60,)


In [6]:
total_rows = len(df_practice)

train_percentage = (
    len(X_train) / total_rows
) * 100

test_percentage = (
    len(X_test) / total_rows
) * 100

print(
    "Training Percentage:",
    round(train_percentage, 2),
    "%"
)

print(
    "Testing Percentage:",
    round(test_percentage, 2),
    "%"
)

Training Percentage: 79.93 %
Testing Percentage: 20.07 %


# Validation Set

We will now divide the data into approximately:

- 70% Training
- 15% Validation
- 15% Testing

The validation set can be used during model development, while the test set remains separate for final evaluation.

In [7]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.1765,
    random_state=42
)

print("Training Rows:", len(X_train))
print("Validation Rows:", len(X_val))
print("Testing Rows:", len(X_test))

Training Rows: 209
Validation Rows: 45
Testing Rows: 45


# Stratified Sampling

Stratified Sampling attempts to preserve target class proportions in the training and testing sets.

In [8]:
print(
    y.value_counts(
        normalize=True
    ).round(3)
)

practice_target
0    0.505
1    0.495
Name: proportion, dtype: float64


In [9]:
X_train_strat, X_test_strat, y_train_strat, y_test_strat = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Distribution:")
print(
    y_train_strat.value_counts(
        normalize=True
    ).round(3)
)

print("\nTesting Distribution:")
print(
    y_test_strat.value_counts(
        normalize=True
    ).round(3)
)

Training Distribution:
practice_target
0    0.506
1    0.494
Name: proportion, dtype: float64

Testing Distribution:
practice_target
0    0.5
1    0.5
Name: proportion, dtype: float64


# Data Leakage Prevention

Preprocessing methods should learn their parameters from the training data only.

For Standard Scaling:

- Fit the scaler on training data.
- Transform training data.
- Use the same fitted scaler to transform test data.

In [10]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train_strat
)

X_test_scaled = scaler.transform(
    X_test_strat
)

print(
    "Scaled Training Shape:",
    X_train_scaled.shape
)

print(
    "Scaled Testing Shape:",
    X_test_scaled.shape
)

Scaled Training Shape: (239, 10)
Scaled Testing Shape: (60, 10)


# Cross Validation

Cross Validation evaluates the model across multiple data folds instead of relying on only one train/test split.

In [11]:
model = LogisticRegression(
    max_iter=5000
)

In [12]:
cv_scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("Fold Scores:")
print(cv_scores)

print(
    "\nAverage Accuracy:",
    round(cv_scores.mean(), 3)
)

Fold Scores:
[0.46666667 0.46666667 0.46666667 0.51666667 0.55932203]

Average Accuracy: 0.495


# Stratified K-Fold

Stratified K-Fold combines Cross Validation with class-proportion preservation.

In [13]:
stratified_kfold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

stratified_scores = cross_val_score(
    model,
    X,
    y,
    cv=stratified_kfold,
    scoring="accuracy"
)

print("Stratified Fold Scores:")
print(stratified_scores)

print(
    "\nAverage Accuracy:",
    round(
        stratified_scores.mean(),
        3
    )
)

Stratified Fold Scores:
[0.36666667 0.43333333 0.51666667 0.48333333 0.45762712]

Average Accuracy: 0.452


In [14]:
summary = pd.DataFrame({
    "Method": [
        "Train/Test Split",
        "Train/Validation/Test",
        "Stratified Split",
        "5-Fold Cross Validation",
        "Stratified 5-Fold"
    ],
    "Main Purpose": [
        "Basic training and testing",
        "Training, tuning and final testing",
        "Preserve class proportions",
        "Evaluate across multiple folds",
        "Cross validation with class balance"
    ]
})

summary

,Method,Main Purpose
0,Train/Test Split,Basic training and testing
1,Train/Validation/Test,"Training, tuning and final testing"
2,Stratified Split,Preserve class proportions
3,5-Fold Cross Validation,Evaluate across multiple folds
4,Stratified 5-Fold,Cross validation with class balance


In [15]:
print("Original Shape:", df.shape)

print(
    "practice_target in Original Dataset:",
    "practice_target" in df.columns
)

Original Shape: (299, 10)
practice_target in Original Dataset: False


# Summary

In this notebook, we practiced:

- Separating Features and Target
- Train/Test Split
- Validation Set
- Stratified Sampling
- Correct Train/Test Scaling
- Cross Validation
- Stratified K-Fold

## Key Learnings

- Training data is used to teach the model.
- Validation data is used during model development.
- Test data is kept separate for final evaluation.
- Stratification helps preserve class proportions.
- Cross Validation evaluates a model across multiple folds.
- Preprocessing should be fitted on training data only.
- Test data should not influence learned preprocessing parameters.

> The `practice_target` used in this notebook is randomly generated and must not be interpreted as a real medical outcome.